In [3]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from typing import TypedDict
from dotenv import load_dotenv


In [5]:

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    provider="featherless-ai",
    task="conversational",
    max_new_tokens=512,
    temperature=0.7,
)
model = ChatHuggingFace(llm=llm)

In [6]:

load_dotenv()
model = ChatHuggingFace(llm=llm)

In [7]:
#create a state
class LLMState(TypedDict):
    question: str
    answer : str

In [8]:
#create the node 
def llm_qa(state: LLMState) -> LLMState:
    #extract the question form state
    question = state['question']
    prompt = f'Answer the following question: {question}'

    #ask the question to the llm 
    answer = model.invoke(prompt).content

    #update the state 
    state['answer'] = answer
    return state

In [9]:
#create the graph
graph = StateGraph(LLMState)

#add nodes
graph.add_node('llm_qa' , llm_qa)

#add edges
graph.add_edge(START , 'llm_qa')
graph.add_edge('llm_qa' , END)

#compile the graph
workflow = graph.compile()

In [10]:
#execute the graph
input_state = {'question': 'What is the capital of France?'}
final_state = workflow.invoke(input_state)
print(final_state['answer'])  

The capital of France is Paris.


In [11]:
model.invoke('What is the capital of France?').content

'The capital of France is Paris.'